# Missing Data Is Not Empty Space
## Statistical analysis of an irregular blood-pressure tracker

This executable notebook treats the **observation process** as part of the statistical problem. The public repository contains only a privacy-safe, day-indexed aggregate snapshot; the private source rows are never committed. This is a statistical case study, not medical advice or a clinical interpretation.

The analysis has evolved beyond one global time slope. The current question is: **which conclusions survive when we challenge the observation process, the long missing gap, individual influential days, the assumed within-episode time form, and temporal dependence under irregular calendar spacing?**

## 1. Load the public calendar grid and tested analysis modules

All statistical computations below come from the same reusable Python modules that are regression-tested in CI. The notebook is the narrative layer, not a second implementation.

In [ ]:
from pathlib import Path
import analysis as primary
import observation_process_sensitivity as observation_sensitivity
import gap_aware_trend_decomposition as gap_aware
import day_influence_sensitivity as influence
import episode_observation_sensitivity as episode_observation
import episode_time_form_sensitivity as time_form
import temporal_dependence_diagnostics as temporal

root = Path.cwd()
data_path = root / 'data/analysis_snapshot.csv'
if not data_path.exists():
    raise FileNotFoundError('Run from Blood_Pressure_Missingness/')
records = primary.load_snapshot(data_path)
observed = primary.observed_records(records)
print(f'{len(records)} calendar days; {len(observed)} observed; {len(records)-len(observed)} missing')

## 2. Observation design first

The current live snapshot has **26 observed days across 65 calendar days**, so **60.0% of days are missing**. One **32-day uninterrupted gap** still accounts for most of that missingness. Sampling intensity is highly uneven, ranging from 1 to 21 readings on an observed day.

That rules out a naïve interpretation in which 171 readings are treated as exchangeable i.i.d. observations from a uniformly observed time series.

## 3. The global HC3 trend is descriptive, not the final story

A day-level linear regression still provides a useful descriptive baseline, but it spans the long unobserved interval.

In [ ]:
global_fit = primary.linear_trend(records, 'mean_systolic_mmHg')
print(
    f"Global systolic trend: {global_fit['slope_per_30_days']:.2f} mmHg/30d, "
    f"95% CI [{global_fit['ci95_low_per_30_days']:.2f}, {global_fit['ci95_high_per_30_days']:.2f}]"
)

## 4. Sensitivity to unequal sampling intensity

The negative global association survives ordinary adjustment and weighting choices. The deliberately aggressive inverse-intensity stress specification remains the exception: its confidence interval crosses zero. This is **not inverse-probability weighting** because the day-observation probabilities are not identified.

In [ ]:
observed_days = observation_sensitivity.load_observed_days(data_path)
obs_results = observation_sensitivity.fit_sensitivity_models(observed_days)
for item in obs_results['trend_estimates']:
    print(
        f"{item['name']}: {item['slope_per_30_days']:.2f} mmHg/30d, "
        f"95% CI [{item['ci95_low_per_30_days']:.2f}, {item['ci95_high_per_30_days']:.2f}]"
    )

## 5. The 32-day gap changes the interpretation

The global slope should not be read as a smooth decline through a period with no measurements. A gap-aware decomposition separates within-episode time variation from the level contrast between the observed episodes on either side of the gap.

In [ ]:
gap_results = gap_aware.analyze_gap_aware_trend(records)
centered = gap_results['episode_centered_model']
decomp = gap_results['exact_global_slope_decomposition']
print(
    f"Post-minus-pre episode contrast: {centered['post_minus_pre_mean_difference_mmHg']:.2f} mmHg, "
    f"95% CI [{centered['post_minus_pre_ci95_low_mmHg']:.2f}, {centered['post_minus_pre_ci95_high_mmHg']:.2f}]"
)
print(
    f"Between-episode share of the global time-pressure covariance: "
    f"{100*decomp['between_fraction_of_time_pressure_covariance']:.1f}%"
)

## 6. No single observed day carries the main result

With 26 observed days, influence still matters. Leave-one-observed-day-out refits hold the full-data episode split fixed and remove each observed day once. The global negative association and the pre/post episode contrast survive every deletion. The **within-episode slope significance remains deletion-sensitive**, even though its current full-data HC3 interval lies just below zero.

In [ ]:
influence_results = influence.summarize_influence(records)
summary = influence_results['leave_one_day_out_summary']
print(
    f"Global slope range after deleting one day: "
    f"[{summary['global_slope_min_per_30_days']:.2f}, {summary['global_slope_max_per_30_days']:.2f}] mmHg/30d"
)
print(
    f"Episode contrast range after deleting one day: "
    f"[{summary['episode_level_difference_min_mmHg']:.2f}, {summary['episode_level_difference_max_mmHg']:.2f}] mmHg"
)
print(
    'Within-episode significant-negative deletions:',
    summary['within_episode_significant_negative_deletions'],
)

## 7. The episode contrast also survives ordinary sampling-intensity adjustments

The gap-defined post-minus-pre level difference remains negative under equal-day, sampling-adjusted, reading-weighted, and capped-weight specifications. Only the aggressive inverse-intensity stress case remains inconclusive.

In [ ]:
episode_obs = episode_observation.fit_episode_observation_sensitivity(records)
for item in episode_obs['estimates']:
    print(
        f"{item['name']}: {item['episode_difference_mmHg']:.2f} mmHg, "
        f"95% CI [{item['episode_ci95_low_mmHg']:.2f}, {item['episode_ci95_high_mmHg']:.2f}]"
    )

## 8. Functional-form sensitivity

On the current live snapshot, the level contrast remains below zero under **all five tested within-episode time forms**, including the five-parameter stress model with separate linear slopes plus a common quadratic term. That stress model is still not a preferred trajectory: the sample remains small and only eight observed days precede the long gap.

In [ ]:
time_results = time_form.fit_episode_time_form_sensitivity(records)
for item in time_results['estimates']:
    print(
        f"{item['name']}: {item['episode_difference_mmHg']:.2f} mmHg, "
        f"95% CI [{item['episode_ci95_low_mmHg']:.2f}, {item['episode_ci95_high_mmHg']:.2f}]"
    )

## 9. Temporal dependence must respect calendar distance

Consecutive observed rows are not equally spaced calendar observations. The current observed-row spacings include 1-, 2-, 3-, and 33-day gaps, so a conventional row-order lag-one residual diagnostic would mix genuinely adjacent days with the pair spanning the long missing interval.

The diagnostic below removes the established gap-aware common-linear episode structure, then compares residual pairs by **exact calendar-day lag within the same gap-defined episode**. Pair counts are shown because this is descriptive small-sample evidence, not a formal serial-independence test.

In [ ]:
temporal_results = temporal.diagnose_temporal_dependence(records)
spacing = temporal_results['observed_order_spacing']
lag_one = next(
    item
    for item in temporal_results['exact_calendar_lag_residual_correlations']
    if item['lag_days'] == 1
)
print('Observed-row spacing counts:', spacing['calendar_gap_days_counts'])
print(
    f"Row-order lag-1 residual correlation: "
    f"{spacing['row_order_lag1_residual_correlation']:.3f}"
)
print(
    f"Exact 1-calendar-day residual correlation: {lag_one['pearson_r']:.3f} "
    f"from {lag_one['n_pairs']} within-episode pairs"
)

## 10. Statistical conclusion

The strongest conclusion is **not** that blood pressure followed a smooth downward trajectory over 65 days. The data do not support that story because the calendar is dominated by a 32-day unobserved interval and the measurement process changes over time.

A more defensible synthesis is:

1. the observed global systolic association is negative;
2. most of that global association is structurally tied to the separation between two observed episodes around the long gap;
3. the post-gap episode sits roughly 5–6 mmHg lower than the pre-gap episode across the tested ordinary specifications;
4. that episode contrast survives single-day deletions, ordinary sampling-intensity adjustments, and all tested within-episode time-form alternatives on the current snapshot;
5. the inverse-intensity sampling stress specification remains inconclusive;
6. residual-dependence diagnostics change when actual calendar spacing is preserved, so row-order adjacency should not be treated as a daily time lag;
7. the data cannot identify **when or why** the level difference arose inside the unobserved interval, nor whether the missingness mechanism is MCAR, MAR, or MNAR.

So the broader lesson remains the same, but sharper: **missing data are part of the statistical process. A credible analysis should stress-test the conclusions created by the observation design rather than erase the gaps and report one smooth line.**